<a href="https://colab.research.google.com/github/LeMaterial/lematerial-llm-synthesis/blob/main/examples/notebooks/tutorials/07_building_a_custom_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7 — Building a custom case study

Tutorial 6 changed *what fields* the pipeline extracts. This one changes *what
domain it is pointed at* — which materials it looks for, which figures it reads,
which numbers it pulls out, and what the output table looks like.

You never edit the pipeline to do this. You assemble a `DomainConfig` from four
pieces and hand it to `BatchRunner`, which already knows how to find PDFs, detect
supplementary files, retry on rate limits, resume interrupted runs and report
progress.

We will build a **thermoelectrics** case study from scratch: extract synthesis
procedures for every composition in a doping series, read the peak figure of
merit *zT* out of the text, and write one flat CSV row per material.

## What you'll learn

1. **`PlotFilterConfig`** — deciding which figures are worth an expensive VLM
   call, and testing that decision without spending anything
2. **The material prompt** — why a vague one quietly destroys everything
   downstream
3. **`BaseTextMetricExtractor`** — pulling a domain-specific number out of the
   paper text
4. **`CsvMasterWriter`** — a custom output schema, tested before any batch runs
5. Assembling all four into a `DomainConfig`, wiring up `BatchRunner`, and
   shipping the result as a runnable script

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- `GEMINI_API_KEY` or `OPENROUTER_API_KEY` for Steps 2 and 3. Steps 1, 4, 5 and 6
  make **no** LLM calls at all.
- **Runtime:** ~15 min. **Cost:** under $0.01 — six short text calls on a
  synthetic one-page paper. No PDFs, no OCR, no vision models.

> The companion reference for this notebook is the
> [Building your own case study](https://lematerial.github.io/lematerial-llm-synthesis/case-studies/custom-domain/)
> guide, which covers the same four pieces as prose plus the full API surface.

## Setup — local or Colab

This notebook runs unchanged in two places:

- **Locally**, from a clone of the repository (`uv sync && uv pip install -e .`),
  with your API keys in the `.env` file at the repository root.
- **On [Google Colab](https://colab.research.google.com)** — click the badge at
  the top. The cell below clones the repository and installs it, which takes a
  few minutes the first time, then reads your keys from Colab's **secret
  manager**: open the 🔑 icon in the left sidebar, add one secret per key
  (`GEMINI_API_KEY`, `OPENROUTER_API_KEY`, …) and switch *Notebook access* on for
  each.

Either way the keys land in `os.environ` and nothing else in the notebook
changes — no key is ever passed as a function argument, so none of them can end
up in the notebook's output or in git.

> If an import fails immediately after the setup cell on Colab, use
> **Runtime → Restart session** and run it again: the clone is cached, so the
> second run is quick.

In [ ]:
# --- Setup: this cell is the only difference between local and Colab -----
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Every key the project knows about. A tutorial only needs a subset; whichever
# ones are missing are reported by the key check further down.
API_KEY_NAMES = (
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "MISTRAL_API_KEY",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
    "HF_TOKEN",
)

if IN_COLAB:
    REPO_URL = "https://github.com/LeMaterial/lematerial-llm-synthesis.git"
    # The installed code has to match this notebook: DomainConfig, BatchRunner
    # and PlotFilterConfig are all read from the clone, so a stale branch means
    # the cells below refer to an API the installed package does not have.
    # Point this at your own branch when running an unmerged tutorial.
    REPO_BRANCH = "main"
    REPO_ROOT = Path("/content/lematerial-llm-synthesis")

    if not REPO_ROOT.exists():
        print(f"Cloning the repository ({REPO_BRANCH}) ...")
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1",
             "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    print("Installing llm-synthesis (a few minutes on the first run) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_ROOT)],
        check=True,
    )

    # Colab keeps secrets outside the notebook, so they cannot leak into its
    # output: add them under the key icon in the left sidebar.
    from google.colab import userdata

    for name in API_KEY_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set, or notebook access not granted - reported below
    KEY_SOURCE = "Colab secrets"
else:
    from dotenv import find_dotenv, load_dotenv

    def find_repo_root(start: Path | None = None) -> Path:
        """Walk up from `start` (default: cwd) until a directory has pyproject.toml."""
        here = (start or Path.cwd()).resolve()
        for candidate in (here, *here.parents):
            if (candidate / "pyproject.toml").exists():
                return candidate
        raise RuntimeError(f"No pyproject.toml found above {here}")

    REPO_ROOT = find_repo_root()
    # find_dotenv walks up from the working directory, so this works whether you
    # started Jupyter at the repo root or inside this folder.
    env_path = find_dotenv(usecwd=True)
    load_dotenv(env_path, override=True)
    KEY_SOURCE = env_path or "no .env found"

print(f"environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f"repo root:   {REPO_ROOT}")
print(f"API keys:    {KEY_SOURCE}")

# Steps 2 and 3 need one of these; Steps 1, 4, 5 and 6 need none.
for name in ("GEMINI_API_KEY", "OPENROUTER_API_KEY"):
    present = "set" if os.environ.get(name) else "MISSING"
    print(f"  {name:<22} {present}")

## The domain, and the shape of the work

**Thermoelectrics.** A paper reports a parent compound and a series of
substitutions, and measures transport properties on each one. The number that
matters is the dimensionless figure of merit *zT*, and its peak value with the
temperature at which it occurs.

None of the three built-in domains fit, so this is a genuine new one. What we
have to supply is exactly four things:

```
        ┌──────────────────── DomainConfig — you write this ────────────────────┐
        │                                                                       │
        │  1 · PlotFilterConfig      which figures reach the VLM                │
        │  2 · Material prompt       what counts as "a material" here           │
        │  3 · Metric extractors     the domain number, from text and/or plots  │
        │  4 · Output writer         what the result files and CSV look like    │
        │                                                                       │
        └───────────────────────────────┬───────────────────────────────────────┘
                                        │
                                        ▼
        ┌──────────────────── BatchRunner — you never edit this ────────────────┐
        │  PDF discovery · SI detection · rate-limit retries · resume · logging │
        │            ↓                                                          │
        │  SynthesisPerformancePipeline                                         │
        │  PDF → materials → synthesis → figures → plot data → linking          │
        └───────────────────────────────────────────────────────────────────────┘
```

The cell below sets up a synthetic one-page thermoelectrics paper to develop
against, and picks the language model. Developing against a small fixed paper
is the right habit: it keeps the loop fast and free, and every prompt problem
you are going to hit shows up on one paper just as clearly as on a hundred.

In [ ]:
import os

from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.llms import SystemPrefixedLM

# Flip this to route every call in the notebook through OpenRouter with a single
# OPENROUTER_API_KEY instead of per-provider keys. Same two-branch helper as in
# every other tutorial that calls a model.
USE_OPENROUTER = False

if USE_OPENROUTER:
    lm = SystemPrefixedLM(
        "",
        "openrouter/google/gemini-3-flash-preview",
        api_base="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        temperature=0.0,
    )
else:
    # Registry keys live in src/llm_synthesis/utils/llms.py. If a model has been
    # retired by its provider you will get a 404 here - pick another key from
    # LLM_REGISTRY, or switch USE_OPENROUTER on.
    lm = get_llm_from_name(
        "gemini-3.0-flash", model_kwargs={"temperature": 0.0}
    )

# A synthetic paper, written for this tutorial. It is deliberately compact but
# has the two features that make thermoelectrics papers awkward: a doping series
# of four compositions, and seven figures of which only three are relevant.
DEMO_PAPER = """# Enhanced thermoelectric performance of Se-substituted Bi2Te3 prepared by ball milling and spark plasma sintering

*This is a synthetic example written for the LeMat-Synth tutorials. It is not a
real publication and the numbers in it are invented.*

## Experimental

High-purity Bi (99.999%), Te (99.999%) and Se (99.99%) shot were weighed in the
stoichiometric ratios required for Bi2Te3, Bi2Te2.7Se0.3 and Bi2Te2.4Se0.6 and
loaded into stainless steel jars in an argon-filled glovebox. The charges were
ball milled at 400 rpm for 8 h using a planetary mill with a ball-to-powder mass
ratio of 15:1.

The resulting powders were consolidated by spark plasma sintering at 400 C for
5 min under a uniaxial pressure of 50 MPa in a graphite die of 12.7 mm diameter,
under vacuum. The heating rate was 50 C/min. Sintered pellets reached a relative
density above 97%.

A Cu-doped variant, Cu0.01Bi2Te2.7Se0.3, was prepared by the identical route with
CuCl (99.99%) added to the initial charge before milling.

Bars of 2 x 2 x 8 mm were cut from the pellets for transport measurements.

## Results and discussion

Figure 1a shows the temperature dependence of the dimensionless figure of merit
zT between 300 and 500 K. Pristine Bi2Te3 reaches a peak zT of 0.72 at 400 K.
Se substitution raises this substantially: Bi2Te2.7Se0.3 attains a peak zT of
1.14 at 425 K, while Bi2Te2.4Se0.6 peaks slightly lower at 0.98, also at 425 K.
The Cu-doped sample Cu0.01Bi2Te2.7Se0.3 shows the best performance overall, with
a peak zT of 1.31 at 450 K.

Figure 1b presents the Seebeck coefficient over the same temperature range. All
compositions show negative Seebeck values consistent with n-type conduction,
with magnitudes between 150 and 220 uV/K at room temperature.

Figure 1c shows the power factor, which peaks near 375 K for every composition.

The lattice thermal conductivity (Figure 2a) decreases monotonically with Se
content, from 0.81 W/m.K for Bi2Te3 to 0.54 W/m.K for Bi2Te2.4Se0.6 at 300 K,
which we attribute to enhanced point-defect phonon scattering. Figure 2b shows
the electrical resistivity, which rises with Se substitution.

Figure 3 shows the powder X-ray diffraction patterns of all four compositions,
all of which index to the rhombohedral R-3m structure with no secondary phases.
Figure 4 shows thermogravimetric traces confirming that no mass loss occurs
below 600 C.
"""

print(f"demo paper: {len(DEMO_PAPER.split())} words")
print(f"model:      {'OpenRouter' if USE_OPENROUTER else lm.model}")

## Step 1 — `PlotFilterConfig`: which figures are worth reading?

Reading a figure costs a vision-model call, and a wrong figure costs more than
that: a thermal-conductivity curve digitised into a `zT` column is worse than no
data. `PlotFilterConfig` decides what gets through, by matching axis labels and
units.

Thermoelectrics has a trait that makes it a good teaching case: **the x-axis
carries no information**. Every plot in the paper is *versus temperature* — zT,
Seebeck, power factor, thermal conductivity, resistivity. Filtering on x alone
would let all five through. The discrimination has to come from the y-axis, and
in particular from the veto list.

Start by writing down the figures you expect to meet. This is just the list from
the demo paper's Results section, and it costs nothing to build:

In [ ]:
from llm_synthesis.models.plot import ExtractedLinePlotData

SERIES = ["Bi2Te3", "Bi2Te2.7Se0.3", "Bi2Te2.4Se0.6", "Cu0.01Bi2Te2.7Se0.3"]


def figure(title, x_label, x_unit, y_label, y_unit):
    """One extracted figure, as the plot extractor would hand it to the filter."""
    return ExtractedLinePlotData(
        # The coordinates are irrelevant to filtering - only the axes matter -
        # but the filter rejects a plot with no series at all, so give it some.
        name_to_coordinates={s: [[300.0, 0.5], [500.0, 1.1]] for s in SERIES},
        title=title,
        x_axis_label=x_label,
        x_axis_unit=x_unit,
        y_left_axis_label=y_label,
        y_left_axis_unit=y_unit,
    )


FIGURES = [
    figure("Fig 1a - zT",                   "Temperature", "K",  "zT",                     ""),
    figure("Fig 1b - Seebeck",              "Temperature", "K",  "Seebeck coefficient",    "uV/K"),
    figure("Fig 1c - power factor",         "Temperature", "K",  "Power factor",           "uW/cm.K2"),
    figure("Fig 2a - thermal conductivity", "Temperature", "K",  "Thermal conductivity",   "W/m.K"),
    figure("Fig 2b - resistivity",          "Temperature", "K",  "Electrical resistivity", "mOhm.cm"),
    figure("Fig 3 - XRD",                   "2theta",      "degree", "Intensity",          "a.u."),
    figure("Fig 4 - TGA",                   "Temperature", "C",  "Weight loss",            "%"),
]

# What we want the filter to decide, written down before we write the filter.
EXPECTED = {
    "Fig 1a - zT": True,
    "Fig 1b - Seebeck": True,
    "Fig 1c - power factor": True,
    "Fig 2a - thermal conductivity": False,
    "Fig 2b - resistivity": False,
    "Fig 3 - XRD": False,
    "Fig 4 - TGA": False,
}

print(f"{len(FIGURES)} figures, of which {sum(EXPECTED.values())} should pass")

Before writing anything, run the **default** filter over that list. The default
is the catalysis configuration — temperature on x, conversion/yield on y — which
is a reasonable guess for a thermoelectrics paper, since the x-axes do match.

`PlotFilter.filter_plots` is the exact code the pipeline runs, so this is not a
simulation:

In [ ]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import PlotFilter


def report(config, figures=FIGURES, expected=EXPECTED):
    """Run the real filter and print a verdict per figure."""
    relevant, skips = PlotFilter(config).filter_plots(figures, log_skipped=False)
    kept = {figures[i].title for i, _ in relevant}

    wrong = 0
    for fig in figures:
        got, want = fig.title in kept, expected[fig.title]
        flag = "ok" if got == want else "<-- WRONG"
        wrong += got != want
        print(f"  {fig.title:<32} kept={str(got):<5} want={str(want):<5} {flag}")

    print(f"\n  kept {len(kept)}/{len(figures)}   skips: {skips}")
    print(f"  {wrong} figure(s) classified wrongly")
    return relevant


print("DEFAULT (catalysis) configuration:\n")
report(PlotFilterConfig())

Every figure is rejected — including the three we want. The default y-axis
keywords are `conversion`, `yield` and `activity`, none of which appear in a
thermoelectrics paper, so nothing survives.

That is the good failure mode. The dangerous one is the opposite: a filter loose
enough to pass `Thermal conductivity` into a column labelled `zT`. So write the
**veto list first**, before the keyword list — the plots that share vocabulary
with the real thing are the ones that will hurt you.

Here `conductivity` is the trap. *Thermal conductivity* and *electrical
conductivity* are both plotted against temperature and both sound like transport
performance, but neither is *zT*:

In [ ]:
thermoelectric_filter = PlotFilterConfig(
    # x-axis: everything in this field is measured against temperature, so this
    # is permissive on purpose. It rejects the XRD plot and nothing else.
    x_axis_labels=["temperature", "temp", "t ("],
    x_axis_units=["k", "°c", "ºc", "c", "kelvin", "celsius"],

    # y-axis: the quantities that ARE thermoelectric performance.
    y_axis_keywords=[
        "zt", "figure of merit", "seebeck", "thermopower",
        "power factor", "thermoelectric",
    ],
    y_axis_units=["uv/k", "µv/k", "uw/cm.k2", "µw/cm·k²", "uw/mk2"],

    # The veto list, written first. Each of these appears on a temperature
    # x-axis in the same paper and must never be mistaken for performance.
    y_axis_exclude_patterns=[
        "thermal conductivity",
        "lattice",
        "resistivity",
        "conductivity",     # catches "electrical conductivity" too
        "carrier",
        "hall",
        "weight",           # TGA
        "intensity",        # XRD / spectra
    ],

    # "%" on its own must not qualify a plot - the TGA weight-loss curve is in
    # percent. Keep the default: a percentage needs a keyword to back it up.
    require_y_keyword_with_percentage=True,
)

print("THERMOELECTRIC configuration:\n")
relevant_plots = report(thermoelectric_filter)

Three kept, four rejected, zero mistakes — and it cost nothing to establish.

> **Keep that check.** The `EXPECTED` dict plus `report()` is a regression test
> for your domain. When you meet a real paper whose figures are classified
> wrongly, add them to `FIGURES` and `EXPECTED`, then fix the config until the
> count returns to zero. This is by far the cheapest loop in the whole project:
> no API keys, no PDFs, instant.

---

## Step 2 — The material prompt

Two strings tell the material extractor what counts as a material here. They are
the highest-leverage text in the whole configuration, because everything
downstream keys off the material list: the synthesis extractor runs once per
material, and the linker can only attach a performance curve to a name that is
already on the list.

Thermoelectrics papers are doping series, which gives an under-specified prompt
two ways to go wrong. It can **collapse** the series into `"Bi2Te3"`, so three
of the four compositions vanish along with their zT values. Or it can be
**over-inclusive** and return the elemental precursors — `Bi`, `Te`, `Se`, the
`CuCl` dopant source — as though they were synthesised products, so the pipeline
spends a synthesis extraction and a metric call on each of them and the CSV
fills up with rows that are not materials.

Run both prompts and see which one you get:

In [ ]:
from llm_synthesis.transformers.material_extraction.dspy_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)


def extract_materials(instructions, output_description):
    """Run the real material extractor with a given pair of prompt strings."""
    extractor = DspyTextExtractor(
        signature=make_dspy_text_extractor_signature(
            instructions=instructions,
            output_name="materials",
            output_description=output_description,
        ),
        lm=lm,
    )
    raw = extractor.forward(input=DEMO_PAPER)
    return [m.strip() for m in raw.split(",") if m.strip()]


vague = extract_materials(
    "Extract the materials in this paper.",
    "The materials.",
)
print("Vague prompt   ->", vague)

In [ ]:
# Specific about three things: every variant separately, exact stoichiometry,
# and precursors explicitly excluded.
MATERIAL_INSTRUCTIONS = (
    "Extract ALL distinct thermoelectric compositions that were SYNTHESIZED "
    "and whose transport properties were measured in this paper. Papers in "
    "this field study a series of substitutions or dopings of one parent "
    "compound: list EACH composition separately, including the pristine "
    "parent and every doped or substituted variant. Do not merge a series "
    "into a single generic formula, and do not list precursors, reagents or "
    "crucible materials."
)

MATERIAL_OUTPUT_DESCRIPTION = (
    "ALL distinct synthesized thermoelectric compositions as a comma-separated "
    "list of chemical formulas with their exact stoichiometric subscripts "
    "(e.g. 'Bi2Te2.7Se0.3'). No commentary."
)

materials = extract_materials(MATERIAL_INSTRUCTIONS, MATERIAL_OUTPUT_DESCRIPTION)
print("Specific prompt ->", materials)

print(f"\n{len(vague)} materials from the vague prompt, {len(materials)} from the specific one.")
print("The paper synthesises 4 compositions: Bi2Te3, Bi2Te2.7Se0.3,")
print("Bi2Te2.4Se0.6 and Cu0.01Bi2Te2.7Se0.3.")

The vague prompt is not *wrong* so much as under-specified — it has no way of
knowing whether you want the doping series enumerated, whether Bi/Te/Se shot
count as materials, or how much stoichiometric precision you need. Every one of
those judgements has to be in the prompt, because there is nowhere else to put
it.

Whichever way it goes wrong, it goes wrong **silently**. Nothing errors and
nothing warns. A collapsed series just produces a smaller CSV, and the missing
rows look exactly like papers that did not report the data. An over-inclusive
list produces a larger one, where `Te` sits in the material column with an empty
zT — and you pay for a synthesis extraction and a metric call on every spurious
entry.

This cell can also come out differently from run to run: it is a language model
following an ambiguous instruction, which is the point. The specific prompt is
the one that returns the same four compositions every time.

> Check the material list on two or three papers before running a batch. It is
> one cheap call per paper and it is the single highest-value review step in
> building a new domain.

---

## Step 3 — A domain metric extractor

The standard pipeline gives you synthesis procedures and digitised curves. It
does not give you *peak zT at temperature T*, because that is domain knowledge.

`BaseTextMetricExtractor` adds one LLM pass over the paper text and returns
`{material_name: {metric: value}}`. There is a matching `BaseVLMMetricProcessor`
for quantities you must read off the *shape* of a curve (an onset, a transition,
an intercept) — we do not need one here, because the demo paper states its peak
zT values in prose, which is where most such numbers actually live.

In [ ]:
from typing import Any

import dspy

from llm_synthesis.domain_metrics.base import BaseTextMetricExtractor


class PeakZTSignature(dspy.Signature):
    """Read the peak thermoelectric figure of merit for one material."""

    paper_text: str = dspy.InputField(description="Full text of the paper.")
    material_name: str = dspy.InputField(
        description="The exact material to report on."
    )

    peak_zT: float | None = dspy.OutputField(
        description=(
            "Highest zT reported for THIS material. Null when the paper does "
            "not state one for it - never infer from another composition."
        )
    )
    T_at_peak_zT_K: float | None = dspy.OutputField(
        description=(
            "Temperature in KELVIN at which peak_zT occurs. Convert from "
            "Celsius if the paper uses it. Null when not stated."
        )
    )
    evidence: str | None = dspy.OutputField(
        description=(
            "The sentence the values were read from, verbatim. Null when not "
            "stated."
        )
    )


class PeakZTExtractor(BaseTextMetricExtractor):
    """One LLM pass per material, returning {material: {metric: value}}."""

    def __init__(self, lm: dspy.LM) -> None:
        self.lm = lm

    def extract(
        self, paper_text: str, materials: list[str]
    ) -> dict[str, Any]:
        results: dict[str, Any] = {}
        with dspy.settings.context(
            lm=self.lm, adapter=dspy.adapters.JSONAdapter()
        ):
            predict = dspy.Predict(PeakZTSignature)
            for material in materials:
                prediction = predict(
                    paper_text=paper_text, material_name=material
                )
                # Materials absent from the returned dict are treated as
                # "no data" downstream, so skip rather than store nulls.
                if prediction.peak_zT is None:
                    continue
                results[material] = {
                    "peak_zT": prediction.peak_zT,
                    "T_at_peak_zT_K": prediction.T_at_peak_zT_K,
                    "evidence": prediction.evidence,
                }
        return results


text_metrics = PeakZTExtractor(lm).extract(DEMO_PAPER, materials)

for material, metric in text_metrics.items():
    print(f"{material:<24} zT={metric['peak_zT']} at {metric['T_at_peak_zT_K']} K")
    print(f"{'':<24} \"{metric['evidence']}\"")

Three details in that signature are doing real work, and all three are the kind
of thing you only learn by reading bad output:

| Choice | Why |
|---|---|
| `float \| None`, not `float` | A required field makes the model invent a number for compositions the paper never characterised |
| *"never infer from another composition"* | Without it, the model happily reports the parent compound's zT for every dopant |
| An `evidence` field | Turns every value into something you can spot-check in seconds, and costs a handful of tokens |

Asking for the temperature **in Kelvin** is the same instinct: half the field
plots in °C, and a column that silently mixes both units is not a column.

---

## Step 4 — The output writer

Two writers ship with the project:

| Writer | Best for | Output |
|---|---|---|
| `AnnotatedJsonWriter` | Rich, qualitative results you will read one at a time | `<output_dir>/<paper_id>/<material>.json` plus linking summaries |
| `CsvMasterWriter` | Tabular data you want to aggregate across papers | The same JSON, **plus** a growing master CSV with one row per (paper, material) |

A thermoelectrics survey is a table — one row per composition, sortable by zT —
so `CsvMasterWriter` with a custom column set is the right choice. Override
`_build_flat_records` to say what a row is:

In [ ]:
from typing import Any

from llm_synthesis.runners.output_writers.csv_writer import CsvMasterWriter

THERMOELECTRIC_COLUMNS = [
    "paper_id",
    "material",
    "synthesis_method",
    "peak_zT",
    "T_at_peak_zT_K",
    "evidence",
]


class ThermoelectricWriter(CsvMasterWriter):
    """Per-material JSON plus a flat thermoelectrics.csv across all papers."""

    def __init__(self) -> None:
        super().__init__(
            csv_columns=THERMOELECTRIC_COLUMNS,
            master_csv_name="thermoelectrics.csv",
        )

    def _build_flat_records(
        self,
        paper_id: str,
        result,                      # PipelineResult
        text_metrics: dict[str, Any],
        vlm_metrics: dict[str, Any],
    ) -> list[dict]:
        rows = []
        for entry in result.results:
            material = entry.material
            metric = text_metrics.get(material, {})
            rows.append(
                {
                    "paper_id": paper_id,
                    "material": material,
                    # entry.synthesis is None when extraction failed for this
                    # material - the row should still exist, with a gap in it.
                    "synthesis_method": (
                        entry.synthesis.synthesis_method
                        if entry.synthesis
                        else None
                    ),
                    "peak_zT": metric.get("peak_zT"),
                    "T_at_peak_zT_K": metric.get("T_at_peak_zT_K"),
                    "evidence": metric.get("evidence"),
                }
            )
        return rows


print("writer ready:", ThermoelectricWriter().__class__.__name__)

Test it now, on a hand-built result, rather than discovering a `KeyError` forty
minutes into a batch. `PipelineResult` is a plain Pydantic model, so you can
construct one without running anything — and this cell makes no LLM calls:

In [ ]:
from llm_synthesis.models.ontologies import GeneralSynthesisOntology
from llm_synthesis.services.pipelines.synthesis_performance_pipeline import (
    PipelineResult,
    SynthesisWithPerformanceEntry,
)

# One material extracted successfully, one where synthesis extraction failed -
# the case that breaks writers that assume entry.synthesis is never None.
fake_result = PipelineResult(
    paper_id="demo_thermoelectric",
    paper_name="demo_thermoelectric.pdf",
    materials=["Bi2Te3", "Cu0.01Bi2Te2.7Se0.3"],
    results=[
        SynthesisWithPerformanceEntry(
            material="Bi2Te3",
            synthesis=GeneralSynthesisOntology(
                target_compound="Bi2Te3",
                target_compound_type="energy & sustainability",
                synthesis_method="ball milling",
            ),
        ),
        SynthesisWithPerformanceEntry(
            material="Cu0.01Bi2Te2.7Se0.3",
            synthesis=None,
        ),
    ],
)

writer = ThermoelectricWriter()
rows = writer._build_flat_records(
    "demo_thermoelectric",
    fake_result,
    text_metrics={"Bi2Te3": {"peak_zT": 0.72, "T_at_peak_zT_K": 400.0,
                             "evidence": "Pristine Bi2Te3 reaches a peak zT of 0.72 at 400 K."}},
    vlm_metrics={},
)

for row in rows:
    print(row)

# And the real write path, into a scratch directory under the git-ignored data/.
scratch = REPO_ROOT / "data" / "tutorials" / "custom_case_study"
summary = writer.write_paper(
    paper_id="demo_thermoelectric",
    output_dir=scratch,
    pipeline_result=fake_result,
    text_metrics={"Bi2Te3": {"peak_zT": 0.72, "T_at_peak_zT_K": 400.0,
                             "evidence": "Pristine Bi2Te3 reaches a peak zT of 0.72 at 400 K."}},
    vlm_metrics={},
    processing_time=12.3,
)
writer.finalize(scratch, [summary])

print("\nfiles written:")
for path in sorted(scratch.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(scratch))

print("\nthermoelectrics.csv:")
print((scratch / "thermoelectrics.csv").read_text())

Both rows are present, and the material whose synthesis extraction failed has a
`None` in `synthesis_method` rather than taking the run down. That is the
behaviour you want in a batch: a partial row is data, an exception is not.

---

## Step 5 — Assemble the `DomainConfig`

All four pieces exist now, so the config is just a constructor call. This is the
whole of what makes a domain:

In [ ]:
from llm_synthesis.config.domain_config import DomainConfig

thermoelectric_domain = DomainConfig(
    name="thermoelectrics",
    # 1 - which figures reach the VLM
    plot_filter_config=thermoelectric_filter,
    # 2 - what counts as a material here
    material_extraction_instructions=MATERIAL_INSTRUCTIONS,
    material_output_description=MATERIAL_OUTPUT_DESCRIPTION,
    # 3 - the domain number (None on the VLM side: zT is stated in prose)
    text_metric_extractor=PeakZTExtractor(lm),
    vlm_metric_processor=None,
    # 4 - what the output looks like
    output_writer=ThermoelectricWriter(),
)

print(f"domain:            {thermoelectric_domain.name}")
print(f"text metrics:      {type(thermoelectric_domain.text_metric_extractor).__name__}")
print(f"vlm metrics:       {thermoelectric_domain.vlm_metric_processor}")
print(f"writer:            {type(thermoelectric_domain.output_writer).__name__}")
print(f"y-axis keywords:   {thermoelectric_domain.plot_filter_config.y_axis_keywords}")
print(f"y-axis vetoes:     {thermoelectric_domain.plot_filter_config.y_axis_exclude_patterns}")

That is exactly the shape of `DomainConfig.for_catalysis()`,
`.for_superconductivity()` and `.for_porosity()` — the built-in factory methods
are convenience wrappers around this same call, nothing more.

---

## Step 6 — Hand it to `BatchRunner`

`BatchRunner` supplies everything that is not domain-specific: PDF discovery,
supplementary-file detection, rate-limit-aware retries, resumable runs and
progress reporting.

The run below is **left commented out on purpose**. It needs a folder of real
PDFs, plus `MISTRAL_API_KEY` for OCR and `ANTHROPIC_API_KEY` for figures, and it
costs real money — on the order of a few cents to a few tens of cents per paper
depending on how many materials each one has.

In [ ]:
from llm_synthesis.runners.batch_runner import BatchRunner

runner = BatchRunner(
    domain_config=thermoelectric_domain,
    gemini_model="gemini-3.0-flash",           # synthesis extraction
    claude_model="claude-sonnet-4-20250514",   # reading figures
    material_model="gemini-3.0-flash",         # the material list
    linker_model="gemini-3.0-flash",           # series -> material linking
    synthesis_max_tokens=80_000,
    linker_max_tokens=32_000,
    max_workers=4,
)

print("runner configured for domain:", runner.domain_config.name)

# --- First run: two papers, no figures. Cheap, fast, and it is where prompt
# --- problems surface. Uncomment when you have PDFs to point it at.
#
# runner.run(
#     pdf_dir=REPO_ROOT / "data" / "papers_thermoelectric",
#     output_dir=REPO_ROOT / "data" / "results_thermoelectric",
#     max_papers=2,
#     skip_figures=True,     # text + synthesis + zT only, no VLM calls
#     skip_existing=True,
# )
#
# --- Then the full run, once the material lists and the CSV look right.
#
# runner.run(
#     pdf_dir=REPO_ROOT / "data" / "papers_thermoelectric",
#     output_dir=REPO_ROOT / "data" / "results_thermoelectric",
#     skip_existing=True,    # resumes an interrupted run
# )

> **Always do the `max_papers=2, skip_figures=True` run first.** It exercises
> OCR, material extraction, synthesis extraction and your text metric in a
> couple of minutes for a few cents — and that is where essentially every prompt
> problem lives. Turning figures on multiplies both cost and runtime by roughly
> an order of magnitude, so earn that first.

---

## Step 7 — Ship it as a script

Notebooks are for building a domain; scripts are for running one. The three
built-in case studies are each about sixty lines with this exact shape, and the
cell below writes yours out in the same form so it can be run from a terminal,
resumed, and put in version control.

In [ ]:
SCRIPT = '''#!/usr/bin/env python3
"""Batch runner for the thermoelectrics case study.

Usage:
    python run.py /path/to/pdfs /path/to/results --skip-existing
    python run.py /path/to/pdfs /path/to/results --max 2 --skip-figures
"""

import argparse
import logging
from pathlib import Path

from dotenv import load_dotenv

from llm_synthesis.config.domain_config import DomainConfig
from llm_synthesis.runners.batch_runner import BatchRunner

# Import the four pieces you developed in Tutorial 7 - keep them in a module
# next to this script rather than pasting them in, so they stay testable.
from thermoelectric_domain import (
    MATERIAL_INSTRUCTIONS,
    MATERIAL_OUTPUT_DESCRIPTION,
    PeakZTExtractor,
    ThermoelectricWriter,
    thermoelectric_filter,
)

load_dotenv(Path(__file__).resolve().parents[3] / ".env", override=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

GEMINI_MODEL = "gemini-3.0-flash"
CLAUDE_MODEL = "claude-sonnet-4-20250514"


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("pdf_dir", help="Directory containing PDF files")
    parser.add_argument("output_dir", help="Output directory")
    parser.add_argument("--max", type=int, default=None, help="Max papers (testing)")
    parser.add_argument("--skip-existing", action="store_true", help="Resume a run")
    parser.add_argument("--skip-figures", action="store_true", help="Text only, no VLM")
    args = parser.parse_args()

    from llm_synthesis.utils.dspy_utils import get_llm_from_name

    lm = get_llm_from_name(GEMINI_MODEL, model_kwargs={"temperature": 0.0})

    domain = DomainConfig(
        name="thermoelectrics",
        plot_filter_config=thermoelectric_filter,
        material_extraction_instructions=MATERIAL_INSTRUCTIONS,
        material_output_description=MATERIAL_OUTPUT_DESCRIPTION,
        text_metric_extractor=PeakZTExtractor(lm),
        vlm_metric_processor=None,
        output_writer=ThermoelectricWriter(),
    )

    BatchRunner(
        domain_config=domain,
        gemini_model=GEMINI_MODEL,
        claude_model=CLAUDE_MODEL,
        material_model=GEMINI_MODEL,
        linker_model=GEMINI_MODEL,
    ).run(
        pdf_dir=args.pdf_dir,
        output_dir=args.output_dir,
        max_papers=args.max,
        skip_existing=args.skip_existing,
        skip_figures=args.skip_figures,
    )


if __name__ == "__main__":
    main()
'''

# Written under the git-ignored data/ folder. Move it to
# examples/scripts/case_study_thermoelectrics/run.py when you want to keep it.
script_path = scratch / "run.py"
script_path.write_text(SCRIPT)

print(f"wrote {script_path}")
print(f"({len(SCRIPT.splitlines())} lines - the whole case study)")

---

## Checklist for a new domain

1. **List the figures you expect**, with a `True`/`False` verdict for each, and
   keep that list as a regression test. Free, instant, and it catches the
   expensive mistakes.
2. **Write the veto list before the keyword list.** False positives come from
   plots that share vocabulary with the real thing.
3. **Check the material list on two or three papers** before running a batch. A
   vague material prompt fails silently and takes the rest of the pipeline with
   it.
4. **Make every metric field optional** and say what "absent" means, or the
   model will invent values.
5. **Ask for one unit** and make the model convert.
6. **Add an evidence field** to anything a human might need to spot-check.
7. **Test the writer against a hand-built `PipelineResult`**, including a
   material whose synthesis is `None`.
8. **First run is `max_papers=2, skip_figures=True`.** Turn figures on once the
   text half is right.

## Where this fits

| Piece | Lives in | Reference |
|---|---|---|
| `PlotFilterConfig` | `llm_synthesis.config.plot_filter_config` | [Configuration API](https://lematerial.github.io/lematerial-llm-synthesis/api/configuration/) |
| `BaseTextMetricExtractor`, `BaseVLMMetricProcessor` | `llm_synthesis.domain_metrics.base` | [Build your own case study](https://lematerial.github.io/lematerial-llm-synthesis/case-studies/custom-domain/) |
| `CsvMasterWriter`, `AnnotatedJsonWriter` | `llm_synthesis.runners.output_writers` | [Output Format](https://lematerial.github.io/lematerial-llm-synthesis/user-guide/output-format/) |
| `DomainConfig`, `BatchRunner` | `llm_synthesis.config`, `llm_synthesis.runners` | [Case Studies](https://lematerial.github.io/lematerial-llm-synthesis/case-studies/) |

## What's next

- **[Tutorial 5 — Evaluating extraction quality](05_evaluating_extraction_quality.ipynb)**:
  your domain is only as good as its output — measure it before you trust the CSV.
- **[Tutorial 6 — Customising the ontology](06_customizing_the_ontology.ipynb)**:
  if the *synthesis* record itself is missing a field your domain needs, that is
  a schema change, not a domain change.
- **[Tutorial 4 — Synthesis + performance from a paper](04_extracting_synthesis_and_performance.ipynb)**:
  what `BatchRunner` is doing per paper, one stage at a time.
- The three shipped domains are the best reference implementations:
  [`case_study_thermocatalysis/`](https://github.com/LeMaterial/lematerial-llm-synthesis/tree/main/examples/scripts/case_study_thermocatalysis),
  [`case_study_superconductors/`](https://github.com/LeMaterial/lematerial-llm-synthesis/tree/main/examples/scripts/case_study_superconductors)
  (the one with a `BaseVLMMetricProcessor`), and
  [`case_study_porosity/`](https://github.com/LeMaterial/lematerial-llm-synthesis/tree/main/examples/scripts/case_study_porosity)
  (the shortest).